# MetroPT-3: Visual Exploratory Data Analysis

**A practical, time-aware exploration of a real railway air-compressor dataset**

MetroPT-3 contains multivariate sensor readings from an Air Production Unit (APU)
installed on a metro train. This notebook goes beyond `head()` and `describe()` to
examine data quality, sampling cadence, operating regimes, sensor relationships,
long-term drift, and four reported air-leak incidents.

### What this notebook covers

1. Memory-conscious loading and schema validation
2. Data quality, duplicates, missingness, cadence, and coverage
3. Analog and digital sensor distributions
4. Correlations and compressor operating regimes
5. Daily trends and pressure relationships
6. Incident-centered visual analysis
7. A transparent, leakage-safe anomaly proxy for exploration

> **Scope:** This is an EDA notebook, not a production safety system. The dataset
represents one APU and only four consolidated incidents. An anomaly is not proof of
a mechanical root cause.

**Data source:** [MetroPT-3 Dataset on Kaggle](https://www.kaggle.com/datasets/joebeachcapital/metropt-3-dataset)  
**Original source:** [UCI MetroPT-3, DOI 10.24432/C5VW3R](https://doi.org/10.24432/C5VW3R)

## 1. Setup

The full CSV has more than 1.5 million rows. We load numeric signals as `float32`
and use deterministic samples only for dense plots. Time aggregations still use the
complete dataset.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
COLORS = {
    "navy": "#16324F",
    "blue": "#247BA0",
    "teal": "#2A9D8F",
    "gold": "#E9C46A",
    "orange": "#F4A261",
    "red": "#E76F51",
    "gray": "#6C757D",
}

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 120,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 2. Load and standardize the data

Kaggle mounts the selected dataset under `/kaggle/input`. The local fallback makes
the same notebook testable outside Kaggle without changing any cell.

In [ ]:
kaggle_root = Path("/kaggle/input")
local_fallback = Path("data/raw/MetroPT3(AirCompressor).csv")
kaggle_csvs = list(kaggle_root.rglob("*.csv")) if kaggle_root.exists() else []

def normalized_filename(path):
    return path.name.lower().replace("_", "").replace("-", "").replace(" ", "")

metropt_candidates = [
    path for path in kaggle_csvs
    if "metropt3" in normalized_filename(path)
]
data_path = metropt_candidates[0] if metropt_candidates else None
if data_path is None and local_fallback.exists():
    data_path = local_fallback
if data_path is None:
    discovered = "\n".join(f"  - {path}" for path in kaggle_csvs[:50])
    raise FileNotFoundError(
        "MetroPT-3 CSV was not found under /kaggle/input. "
        "Use Add Input and select joebeachcapital/metropt-3-dataset."
        + (f"\nCSV files currently visible:\n{discovered}" if discovered else "\nNo CSV inputs are mounted.")
    )

df = pd.read_csv(data_path, low_memory=False)
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)
df = df.rename(columns={"dv_eletric": "dv_electric"})
df = df.drop(
    columns=[
        column for column in df.columns
        if column.startswith("unnamed") or column in {"index", "level_0"}
    ],
    errors="ignore",
)

ANALOG = [
    "tp2", "tp3", "h1", "dv_pressure", "reservoirs",
    "oil_temperature", "motor_current",
]
DIGITAL = [
    "comp", "dv_electric", "towers", "mpg", "lps",
    "pressure_switch", "oil_level", "caudal_impulses",
]
SENSORS = ANALOG + DIGITAL
required = {"timestamp", *SENSORS}
missing_columns = sorted(required.difference(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")
for column in SENSORS:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype("float32")

df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 3. Dataset at a glance

The table below separates analog measurements from binary control/state signals.
Timestamp is used for ordering and validation, not as a predictive sensor.

In [ ]:
overview = pd.DataFrame({
    "value": [
        f"{len(df):,}",
        df["timestamp"].min(),
        df["timestamp"].max(),
        len(ANALOG),
        len(DIGITAL),
        f"{df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB",
    ]
}, index=[
    "Rows", "Start", "End", "Analog sensors", "Digital sensors", "Memory usage"
])
display(overview)
display(df.head(5).style.set_caption("First five standardized records"))

### Sensor dictionary

| Signal | Type | Interpretation |
|---|---|---|
| TP2 | Analog | Pressure at the compressor |
| TP3 | Analog | Pressure at the pneumatic panel |
| H1 | Analog | Pressure associated with the cyclonic-separator discharge |
| DV pressure | Analog | Pressure drop during air-dryer tower discharge |
| Reservoirs | Analog | Downstream reservoir pressure; expected to track TP3 |
| Oil temperature | Analog | Compressor oil temperature |
| Motor current | Analog | Motor state/load proxy |
| COMP | Digital | Compressor intake-valve electrical signal |
| DV electric | Digital | Compressor outlet-valve control signal |
| TOWERS | Digital | Active air-dryer tower |
| MPG | Digital | Compressor load-start control signal |
| LPS | Digital | Low-pressure signal, active below approximately 7 bar |
| Pressure switch | Digital | Air-dryer discharge detection |
| Oil level | Digital | Low-oil-level indication |
| Caudal impulses | Digital | Airflow pulse signal from APU to reservoirs |

## 4. Data-quality audit

Missing values are only one part of time-series quality. Duplicate timestamps and
irregular gaps can be equally important because they distort rolling and sequence
models.

In [ ]:
duplicate_timestamps = int(df["timestamp"].duplicated().sum())
missing = df.isna().sum().sort_values(ascending=False)
digital_validity = pd.DataFrame({
    "unique_values": [sorted(df[c].dropna().unique().tolist()) for c in DIGITAL],
    "is_binary": [set(df[c].dropna().unique()).issubset({0.0, 1.0}) for c in DIGITAL],
}, index=DIGITAL)

quality_summary = pd.DataFrame({
    "metric": [
        "Missing sensor values",
        "Duplicate timestamps",
        "Rows out of chronological order before sorting",
        "Digital signals with non-binary values",
    ],
    "count": [
        int(df[SENSORS].isna().sum().sum()),
        duplicate_timestamps,
        0,
        int((~digital_validity["is_binary"]).sum()),
    ],
})
display(quality_summary)
display(digital_validity)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
missing_pct = (missing / len(df) * 100).reindex(df.columns)
axes[0].bar(missing_pct.index, missing_pct.values, color=COLORS["blue"])
axes[0].set_title("Missing values by column")
axes[0].set_ylabel("Missing (%)")
axes[0].tick_params(axis="x", rotation=75)

unique_counts = df[SENSORS].nunique().sort_values()
axes[1].barh(unique_counts.index, unique_counts.values, color=COLORS["teal"])
axes[1].set_xscale("log")
axes[1].set_title("Number of unique values (log scale)")
axes[1].set_xlabel("Unique values")
plt.tight_layout()
plt.show()

## 5. Sampling cadence and temporal coverage

The UCI metadata contains both 1 Hz and 0.1 Hz descriptions. Rather than assuming a
perfect clock, we inspect the published timestamps directly. Longer gaps matter for
sequence construction even when the CSV has no explicit null values.

In [ ]:
cadence_seconds = df["timestamp"].diff().dt.total_seconds().dropna()
cadence_stats = cadence_seconds.describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
).to_frame("seconds")
display(cadence_stats)

daily_counts = df.set_index("timestamp").resample("1D").size()
cadence_cap = max(20, cadence_seconds.quantile(0.995))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(
    cadence_seconds.clip(upper=cadence_cap),
    bins=80,
    color=COLORS["blue"],
    alpha=0.9,
)
axes[0].axvline(cadence_seconds.median(), color=COLORS["red"], ls="--", lw=2,
                label=f"Median = {cadence_seconds.median():.1f}s")
axes[0].set_title("Observed interval between consecutive records")
axes[0].set_xlabel(f"Seconds (clipped at 99.5th percentile: {cadence_cap:.0f}s)")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].plot(daily_counts.index, daily_counts.values, color=COLORS["navy"], lw=1.4)
axes[1].axhline(daily_counts.median(), color=COLORS["orange"], ls="--",
                label=f"Median = {daily_counts.median():,.0f} rows/day")
axes[1].set_title("Daily observation count")
axes[1].set_ylabel("Rows")
axes[1].legend()
axes[1].xaxis.set_major_locator(mdates.MonthLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

## 6. Analog sensor distributions

Compressor telemetry is strongly multimodal: the machine cycles between off,
offloaded, loaded, and start-up regimes. For plotting speed, the following charts use
a reproducible sample while retaining the original values.

In [ ]:
plot_sample = df.sample(n=min(150_000, len(df)), random_state=SEED)

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()
for axis, sensor in zip(axes, ANALOG):
    axis.hist(
        plot_sample[sensor].dropna(), bins=80,
        color=COLORS["blue"], alpha=0.82,
    )
    axis.axvline(plot_sample[sensor].median(), color=COLORS["red"], ls="--", lw=1.8)
    axis.set_title(sensor.replace("_", " ").title())
    axis.set_ylabel("Sample count")
for axis in axes[len(ANALOG):]:
    axis.set_visible(False)
fig.suptitle("Analog sensor distributions (deterministic sample)", fontsize=17, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
box_sample = plot_sample.sample(n=min(40_000, len(plot_sample)), random_state=SEED)
analog_median = box_sample[ANALOG].median()
analog_mad = (box_sample[ANALOG] - analog_median).abs().median().replace(0, np.nan)
robust_scaled = ((box_sample[ANALOG] - analog_median) / (1.4826 * analog_mad)).clip(-8, 8)
robust_long = robust_scaled.melt(var_name="sensor", value_name="robust_z")

plt.figure(figsize=(14, 6))
sns.boxplot(
    data=robust_long, x="sensor", y="robust_z",
    color=COLORS["teal"], showfliers=False,
)
plt.axhline(0, color="black", lw=1)
plt.title("Analog sensors on a comparable robust scale")
plt.xlabel("")
plt.ylabel("Robust z-score (clipped to ±8 for display)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 7. Digital operating signals

For binary sensors, the mean equals the fraction of time the signal is active. Weekly
averages reveal whether operating logic or duty cycle changes across the observation
period.

In [ ]:
active_rate = df[DIGITAL].mean().sort_values()
weekly_digital = df.set_index("timestamp")[DIGITAL].resample("7D").mean().T

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
axes[0].barh(active_rate.index, active_rate.values * 100, color=COLORS["teal"])
axes[0].set_title("Overall digital-signal active rate")
axes[0].set_xlabel("Active observations (%)")
for index, value in enumerate(active_rate.values * 100):
    axes[0].text(value + 0.5, index, f"{value:.1f}%", va="center", fontsize=9)

sns.heatmap(
    weekly_digital,
    cmap="YlGnBu", vmin=0, vmax=1,
    cbar_kws={"label": "Weekly active ratio"}, ax=axes[1],
)
axes[1].set_title("Weekly digital-signal activity")
axes[1].set_xlabel("Week index")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

## 8. Sensor relationships

Correlation is useful for finding redundant or physically coupled signals, but it is
not a causal explanation. A random sample is sufficient here because correlation does
not require every highly autocorrelated raw row.

In [ ]:
corr_sample = plot_sample[SENSORS].sample(
    n=min(100_000, len(plot_sample)), random_state=SEED
)
correlation = corr_sample.corr(method="spearman")

mask = np.triu(np.ones_like(correlation, dtype=bool))
plt.figure(figsize=(13, 10))
sns.heatmap(
    correlation, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.4, cbar_kws={"shrink": 0.8, "label": "Spearman ρ"},
)
plt.title("Spearman correlation between sensor signals")
plt.tight_layout()
plt.show()

pairs = (
    correlation.where(~np.eye(len(correlation), dtype=bool))
    .stack()
    .rename("rho")
    .reset_index()
    .rename(columns={"level_0": "sensor_a", "level_1": "sensor_b"})
)
pairs["abs_rho"] = pairs["rho"].abs()
pairs = pairs[pairs["sensor_a"] < pairs["sensor_b"]]
display(pairs.nlargest(12, "abs_rho").drop(columns="abs_rho"))

## 9. Compressor operating regimes

Motor current is a useful proxy for compressor state. The published documentation
describes values near 0 A (off), 4 A (offloaded), 7 A (loaded), and 9 A (start-up).
The boundaries below are exploratory approximations, not maintenance rules.

In [ ]:
operating_sample = plot_sample[["motor_current", "tp2", "tp3", "reservoirs"]].dropna()
operating_sample = operating_sample.assign(
    state=pd.cut(
        operating_sample["motor_current"],
        bins=[-np.inf, 1.0, 5.5, 8.2, np.inf],
        labels=["Off", "Offloaded", "Loaded", "Start / high current"],
    ),
    tp3_minus_reservoirs=lambda frame: frame["tp3"] - frame["reservoirs"],
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(operating_sample["motor_current"], bins=100, color=COLORS["navy"])
for value, label in [(0, "off"), (4, "offloaded"), (7, "loaded"), (9, "start")]:
    axes[0].axvline(value, color=COLORS["red"], alpha=0.55, ls="--")
    axes[0].text(value, axes[0].get_ylim()[1] * 0.86, label, rotation=90, va="top")
axes[0].set_title("Motor-current operating modes")
axes[0].set_xlabel("Motor current (A)")

hex_plot = axes[1].hexbin(
    operating_sample["tp2"], operating_sample["tp3"],
    C=operating_sample["motor_current"], reduce_C_function=np.mean,
    gridsize=55, mincnt=5, cmap="viridis",
)
axes[1].set_title("TP2 vs TP3, colored by mean motor current")
axes[1].set_xlabel("TP2 (bar)")
axes[1].set_ylabel("TP3 (bar)")
fig.colorbar(hex_plot, ax=axes[1], label="Mean motor current (A)")

sns.boxplot(
    data=operating_sample, x="state", y="tp3_minus_reservoirs",
    showfliers=False, color=COLORS["gold"], ax=axes[2],
)
axes[2].set_title("Panel-to-reservoir pressure gap by regime")
axes[2].set_xlabel("")
axes[2].set_ylabel("TP3 - Reservoirs (bar)")
axes[2].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 10. Long-term trends

Daily medians reduce within-cycle noise and expose slow changes. The shaded band is the
daily interquartile range, so it also shows how operating variability evolves.

In [ ]:
time_indexed = df.set_index("timestamp")
daily_median = time_indexed[ANALOG].resample("1D").median()
daily_q25 = time_indexed[ANALOG].resample("1D").quantile(0.25)
daily_q75 = time_indexed[ANALOG].resample("1D").quantile(0.75)

fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True)
axes = axes.ravel()
for axis, sensor in zip(axes, ANALOG):
    axis.plot(daily_median.index, daily_median[sensor], color=COLORS["navy"], lw=1.5)
    axis.fill_between(
        daily_median.index,
        daily_q25[sensor].to_numpy(), daily_q75[sensor].to_numpy(),
        color=COLORS["blue"], alpha=0.22, label="Daily IQR",
    )
    axis.set_title(sensor.replace("_", " ").title())
    axis.xaxis.set_major_locator(mdates.MonthLocator())
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
axes[-1].set_visible(False)
fig.suptitle("Daily analog-sensor median and interquartile range", fontsize=17, y=1.0)
plt.tight_layout()
plt.show()

## 11. Five-minute causal view

Raw 10-second observations are too granular for incident-level interpretation. We now
create right-closed five-minute summaries using only data available up to each bin end.
A full bin should contain roughly 30 readings; bins below 80% coverage are retained for
visualization but marked as low quality.

In [ ]:
five_minute = time_indexed[SENSORS].resample("5min", label="right", closed="right").mean()
five_minute["sample_count"] = time_indexed.resample(
    "5min", label="right", closed="right"
).size()
five_minute["coverage_ratio"] = (five_minute["sample_count"] / 30).clip(upper=1)
five_minute["quality_ok"] = five_minute["coverage_ratio"] >= 0.80
five_minute["tp3_minus_reservoirs"] = (
    five_minute["tp3"] - five_minute["reservoirs"]
)
five_minute["tp2_minus_tp3"] = five_minute["tp2"] - five_minute["tp3"]

coverage_summary = five_minute["quality_ok"].value_counts().rename(
    index={True: "Adequate coverage", False: "Below 80% coverage"}
).to_frame("five_minute_bins")
display(coverage_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(five_minute["coverage_ratio"], bins=40, color=COLORS["teal"])
axes[0].axvline(0.8, color=COLORS["red"], ls="--", label="80% quality gate")
axes[0].set_title("Five-minute data coverage")
axes[0].set_xlabel("Coverage ratio")
axes[0].legend()

pressure_deltas = five_minute[["tp3_minus_reservoirs", "tp2_minus_tp3"]].dropna()
sns.histplot(
    pressure_deltas, element="step", fill=False, bins=80,
    palette=[COLORS["blue"], COLORS["orange"]], ax=axes[1],
)
axes[1].set_title("Engineered pressure differences")
axes[1].set_xlabel("Pressure difference (bar)")
plt.tight_layout()
plt.show()

## 12. Reported air-leak incidents

MetroPT-3 is not labeled row by row. Instead, the source provides four consolidated
high-stress air-leak intervals. These annotations are sparse and should be used for
event-level evaluation rather than treating every raw row as independent evidence.

In [ ]:
events = pd.DataFrame([
    ("Event 1", "2020-04-18 00:00", "2020-04-18 23:59", "Air leak / high stress"),
    ("Event 2", "2020-05-29 23:30", "2020-05-30 06:00", "Air leak / high stress"),
    ("Event 3", "2020-06-05 10:00", "2020-06-07 14:30", "Air leak / high stress"),
    ("Event 4", "2020-07-15 14:30", "2020-07-15 19:00", "Air leak / high stress"),
], columns=["event", "start", "end", "condition"])
events[["start", "end"]] = events[["start", "end"]].apply(pd.to_datetime)
events["duration_hours"] = (
    events["end"] - events["start"]
).dt.total_seconds() / 3600
display(events)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].plot(
    daily_median.index, daily_median["motor_current"],
    color=COLORS["navy"], label="Daily median motor current",
)
axes[1].plot(
    daily_median.index, daily_median["tp3"],
    color=COLORS["blue"], label="TP3",
)
axes[1].plot(
    daily_median.index, daily_median["reservoirs"],
    color=COLORS["orange"], label="Reservoirs",
)
for event_number, event in events.iterrows():
    for axis in axes:
        axis.axvspan(
            event["start"], event["end"], color=COLORS["red"], alpha=0.24,
            label="Reported incident" if event_number == 0 else None,
        )
axes[0].set_title("Reported incidents against daily motor-current behavior")
axes[1].set_title("Reported incidents against daily pressure behavior")
axes[0].legend(loc="upper right")
axes[1].legend(loc="upper right")
axes[1].xaxis.set_major_locator(mdates.MonthLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

## 13. Incident-centered sensor trajectories

To compare sensors with different units, each five-minute value is robustly normalized
using only the **1–21 February reference period**. Each panel is aligned so incident
start equals hour zero. This avoids using future incidents to define the baseline.

In [ ]:
reference_mask = five_minute.index.to_series().between(
    pd.Timestamp("2020-02-01"), pd.Timestamp("2020-02-21 23:59:59")
) & five_minute["quality_ok"]
reference = five_minute.loc[reference_mask, SENSORS]
reference_median = reference.median()
reference_mad = (reference - reference_median).abs().median().replace(0, np.nan)

focus_sensors = ["motor_current", "tp2", "tp3", "reservoirs"]
normalized_five = (
    (five_minute[focus_sensors] - reference_median[focus_sensors])
    / (1.4826 * reference_mad[focus_sensors])
).clip(-8, 8)

fig, axes = plt.subplots(4, 1, figsize=(16, 15), sharex=True, sharey=True)
palette = [COLORS["navy"], COLORS["red"], COLORS["blue"], COLORS["orange"]]
for axis, (_, event) in zip(axes, events.iterrows()):
    window_start = event["start"] - pd.Timedelta(hours=48)
    window_end = event["start"] + pd.Timedelta(hours=12)
    window = normalized_five.loc[window_start:window_end]
    relative_hours = (window.index - event["start"]).total_seconds() / 3600
    for sensor, color in zip(focus_sensors, palette):
        axis.plot(relative_hours, window[sensor], lw=1.0, alpha=0.9,
                  color=color, label=sensor)
    axis.axvspan(-24, -2, color=COLORS["gold"], alpha=0.18,
                 label="24h–2h early-warning window")
    axis.axvline(0, color=COLORS["red"], lw=2, ls="--", label="Incident start")
    axis.set_title(f"{event['event']}: {event['start']:%Y-%m-%d %H:%M}")
    axis.set_ylabel("Robust z-score")
axes[0].legend(ncol=3, loc="upper left")
axes[-1].set_xlabel("Hours relative to incident start")
fig.suptitle("Sensor behavior around reported incidents", fontsize=17, y=1.0)
plt.tight_layout()
plt.show()

## 14. Reference, early-warning, and incident profiles

The next heatmap compares median signed deviations from the February reference. It is
descriptive: with only four events, the apparent differences are hypotheses for future
validation, not universal failure signatures.

In [ ]:
robust_five = (
    (five_minute[SENSORS] - reference_median)
    / (1.4826 * reference_mad)
).replace([np.inf, -np.inf], np.nan)

group = pd.Series("Other holdout", index=five_minute.index, dtype="object")
group.loc[reference_mask] = "Reference (Feb 1–21)"
for _, event in events.iterrows():
    group.loc[event["start"] - pd.Timedelta(hours=24):
              event["start"] - pd.Timedelta(hours=2)] = "Early warning (24h–2h)"
    group.loc[event["start"]:event["end"]] = "Reported incident"

selected_groups = ["Reference (Feb 1–21)", "Early warning (24h–2h)", "Reported incident"]
profile = robust_five.assign(period=group).groupby("period")[SENSORS].median()
profile = profile.reindex(selected_groups)

plt.figure(figsize=(15, 4.5))
sns.heatmap(
    profile, cmap="vlag", center=0, robust=True,
    annot=True, fmt=".1f", cbar_kws={"label": "Median robust deviation"},
)
plt.title("Median sensor profile by operational period")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 15. Leakage-safe exploratory anomaly proxy

This final visual is intentionally simple and transparent:

- Fit feature medians and MADs on **1–21 February only**.
- Compute each five-minute row's robust deviations.
- Average the three largest absolute deviations.
- Normalize and set the threshold using **22–29 February only**.
- Display March–September without retuning after seeing incidents.

This is not presented as a production model. It demonstrates why chronological splits
and a separate threshold-calibration period matter in predictive-maintenance research.

In [ ]:
model_columns = SENSORS + ["tp3_minus_reservoirs", "tp2_minus_tp3"]
train_mask = five_minute.index.to_series().between(
    pd.Timestamp("2020-02-01"), pd.Timestamp("2020-02-21 23:59:59")
) & five_minute["quality_ok"]
calibration_mask = five_minute.index.to_series().between(
    pd.Timestamp("2020-02-22 01:00"), pd.Timestamp("2020-02-29 23:59:59")
) & five_minute["quality_ok"]
test_mask = (five_minute.index >= pd.Timestamp("2020-03-01 01:00")) & five_minute["quality_ok"]

train_frame = five_minute.loc[train_mask, model_columns]
train_center = train_frame.median()
train_mad = (train_frame - train_center).abs().median().replace(0, np.nan)

deviation = (
    (five_minute[model_columns] - train_center).abs()
    / (1.4826 * train_mad)
).replace([np.inf, -np.inf], np.nan).fillna(0)
values = deviation.to_numpy()
anomaly_score = pd.Series(
    np.partition(values, -3, axis=1)[:, -3:].mean(axis=1),
    index=five_minute.index,
    name="raw_score",
)

calibration_scores = anomaly_score.loc[calibration_mask]
score_median = calibration_scores.median()
score_mad = (calibration_scores - score_median).abs().median()
normalized_score = (anomaly_score - score_median) / max(1.4826 * score_mad, 1e-9)
smoothed_score = normalized_score.ewm(alpha=0.2, adjust=False).mean()
threshold = smoothed_score.loc[calibration_mask].quantile(0.995)

display(pd.DataFrame({
    "split": ["Train", "Calibration", "Locked display period"],
    "start": ["2020-02-01", "2020-02-22 01:00", "2020-03-01 01:00"],
    "purpose": ["Fit reference", "Normalize score and lock threshold", "Visualize only"],
    "valid_bins": [int(train_mask.sum()), int(calibration_mask.sum()), int(test_mask.sum())],
}))

display_score = smoothed_score.loc[test_mask].resample("30min").max()
fig, axis = plt.subplots(figsize=(17, 6))
axis.plot(display_score.index, display_score, color=COLORS["navy"], lw=0.8,
          label="30-minute max of smoothed robust score")
axis.axhline(threshold, color=COLORS["red"], ls="--", lw=2,
             label=f"Locked calibration threshold = {threshold:.2f}")
for event_number, event in events.iterrows():
    axis.axvspan(
        event["start"] - pd.Timedelta(hours=24),
        event["start"] - pd.Timedelta(hours=2),
        color=COLORS["gold"], alpha=0.15,
        label="24h–2h early-warning window" if event_number == 0 else None,
    )
    axis.axvspan(
        event["start"], event["end"], color=COLORS["red"], alpha=0.22,
        label="Reported incident" if event_number == 0 else None,
    )
axis.set_title("Exploratory anomaly score on the untouched March–September period")
axis.set_ylabel("Normalized, EWMA-smoothed score")
axis.set_xlabel("")
axis.legend(ncol=2, loc="upper left")
axis.xaxis.set_major_locator(mdates.MonthLocator())
axis.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

## 16. Key takeaways

1. **The system is regime-driven.** Pressure, motor current, and valve signals form
   distinct operating modes; a single global Gaussian assumption would be unrealistic.
2. **Timestamp quality matters.** Even without explicit nulls, gaps and low-coverage
   windows can invalidate sequence features.
3. **Pressure relationships are informative.** TP2, TP3, and reservoir pressure should
   be analyzed jointly rather than as isolated columns.
4. **The dataset is highly autocorrelated.** Millions of raw rows do not equal millions
   of independent examples.
5. **Only four incidents are available.** Event-level recall, lead time, false alarms per
   day, and PR-AUC are more informative than raw pointwise accuracy.
6. **Anomaly explanations are not root causes.** A sensor that drives an anomaly score
   identifies where behavior changed, not why the mechanical failure occurred.

### Recommended next steps

- Build causal five-minute statistical features and 60-minute sequences.
- Keep February for reference learning and threshold calibration.
- Compare transparent robust baselines with PCA, Isolation Forest, and an autoencoder.
- Evaluate once on a chronologically locked March–September holdout.
- Report every incident separately and include false-alarm burden.

---

If this notebook was useful, consider upvoting it and reviewing the original UCI data
card before drawing operational conclusions.